<a href="https://colab.research.google.com/github/Anmolsharma44/tndrr-ai/blob/main/Task_2_RAG_Pipeline/A%5B2%5DRag_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu numpy

Step 2: PDF Load + Custom Chunking Function (From Scratch)
Yeh code PDF ke har page se text nikalega aur usko word-count ke basis par Small Chunks (150 words) aur Large Chunks (600 words) mein overlap ke saath todega:

In [ ]:
import re
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# 1. PDF se text extract karna (page number track karte hue)
def load_pdf_pages(pdf_path):
    reader = PdfReader(pdf_path)
    print(f"Total Pages in PDF: {len(reader.pages)}")
    assert len(reader.pages) >= 21, "PDF kam se kam 30 pages ki honi chahiye!"

    pages = []
    for idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r'\s+', ' ', text).strip()  # extra spaces clean karo
        if text:
            pages.append({"page": idx + 1, "text": text})
    return pages

# 2. Pure Python sliding-window chunker (bina kisi framework ke)
def create_chunks(pages, chunk_size=150, overlap=30):
    chunks = []
    for p in pages:
        words = p["text"].split()
        step = chunk_size - overlap
        for i in range(0, len(words), step):
            chunk_words = words[i : i + chunk_size]
            if len(chunk_words) < 20:  # bahut chhote trailing fragments skip karo
                continue
            chunks.append({
                "page": p["page"],
                "text": " ".join(chunk_words)
            })
    return chunks

# Apni PDF ka path yahan daalo
pages = load_pdf_pages("/content/pdf_to_work.pdf")

small_chunks = create_chunks(pages, chunk_size=150, overlap=30)
large_chunks = create_chunks(pages, chunk_size=600, overlap=100)

print(f"Small chunks count: {len(small_chunks)}")
print(f"Large chunks count: {len(large_chunks)}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/pdf_to_work.pdf'

Step 3: Embeddings + FAISS Index (From Scratch)
Yahan hum BAAI/bge-small-en-v1.5 (fast aur high-accuracy embedding model) use karke vectors banayenge aur Cosine Similarity ke liye IndexFlatIP (Inner Product on normalized vectors) banayenge

========================================
• BAAI/bge-small-en-v1.5 (Translator / Encoder). Iska kaam sirf text ko numbers (vectors) mein convert karna hai. Yeh ek chunk ko padhta hai aur uske semantic meaning ko 384 numbers ki list bana deta hai. Yeh kisi doosre chunk se compare nahi karta.

===============================
• FAISS (Search & Ranking Engine). Jab saare chunks vectors ban jaate hain, toh unhe FAISS mein store kiya jaata hai. Jab user koi sawal (query) puchta hai, toh FAISS us sawal ke vector aur baaki sab chunks ke vectors ke beech distance ya similarity score calculate karta hai, aur sabse best matches ko top par rank karke nikalta hai.

========================================================================================
• One-Hot Encoding (Sirf Ginti). Yeh sirf 0 aur 1 use karta hai. Isme words ka koi matlab nahi hota, bas yeh batata hai ki word dictionary mein kahan hai.
=======================
• Word2Vec (Purana Meaning Model). Yeh har ek single word ko vector banata hai. Isme ek problem thi — "Apple" (phal) aur "Apple" (company) dono ka vector ek hi banta tha, kyunki yeh poora sentence nahi padhta tha.
=========================
• BGE (Modern Context Model). Yeh Word2Vec ka advanced roop hai. Yeh poore sentence ya chunk ko ek saath padhta hai aur uske context ke hisab se vector banata hai. Isliye ise "Apple khana sehat ke liye achha hai" aur "Apple ne naya iPhone launch kiya" ke beech ka farq samajh aata hai.

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

def build_faiss_index(chunks):
    texts = [c["text"] for c in chunks]
    # normalize_embeddings=True karne se Inner Product == Cosine Similarity ban jaata hai
    embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    embeddings = np.array(embeddings, dtype="float32")

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

small_index = build_faiss_index(small_chunks)
large_index = build_faiss_index(large_chunks)

def retrieve_top_k(query, index, chunks, k=5):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    q_emb = np.array(q_emb, dtype="float32")
    scores, indices = index.search(q_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"]
        })
    return results

In [ ]:
import re

def clean_pdf_text(text):
    text = re.sub(r'-\s+', '', text)   # line-break hyphens join karo
    text = re.sub(r'\s+', ' ', text)    # extra spaces/newlines clean karo
    return text.strip()

# pdf_to_work.pdf (NVAWS Report) ke exact sentences par based 15 questions aur source passages
eval_set = [
    {
        "question": "How many women and men were interviewed in the National Violence Against Women Survey (NVAWS)?",
        "source_passage": "This nationally representative telephone survey asked 8,000 women and 8,000 men about their experiences as rape victims."
    },
    {
        "question": "What proportion of women in the United States have been raped at some time in their lives according to NVAWS?",
        "source_passage": "17.6 percent of surveyed women and 3 percent of surveyed men were raped at some time in their lives"
    },
    {
        "question": "What were the primary reasons women chose not to report their adult rape victimization to the police?",
        "source_passage": "Fear of their rapist, embarrassment, and not considering their rape a crime or police matter were the primary reasons women chose not to report their victimization to the police."
    },
    {
        "question": "How was rape defined in the National Violence Against Women Survey?",
        "source_passage": "rape was defined as an event that occurred without the victim's consent that involved the use or threat of force in vaginal, anal, or oral intercourse."
    },
    {
        "question": "How did NVAWS select the designated respondent in households with more than one eligible adult?",
        "source_passage": "In households with more than one eligible adult, the adult with the most recent birthday was selected as the designated respondent."
    },
    {
        "question": "How many rapes were estimated to be perpetrated against U.S. women and men in 1995?",
        "source_passage": "an estimated 876,064 rapes were perpetrated against U.S. women and an estimated 111,298 rapes were perpetrated against U.S. men in 1995"
    },
    {
        "question": "Why are NVAWS annual rape estimates higher than those of the National Crime Victimization Survey (NCVS)?",
        "source_passage": "NVAWS uses five questions to screen respondents for rape victimization (see \"Rape Screening Questions\"), while NCVS uses only two questions."
    },
    {
        "question": "What percentage of female and male rape victims were raped by just one person over their lifetimes?",
        "source_passage": "Among female rape victims, 78.2 percent were raped by one person, 13.5 percent were raped by two people, and 8.3 percent were raped by three or more people"
    },
    {
        "question": "Which racial or ethnic group of women reported significantly higher lifetime rates of rape victimization than other backgrounds?",
        "source_passage": "American Indian/Alaska Native women were significantly more likely than women from all other backgrounds to have been raped at some time in their lifetime"
    },
    {
        "question": "What percentage of female and male rape victims were first raped before their 18th birthday?",
        "source_passage": "more than half (54 percent) of female victims and nearly three-quarters (71 percent) of male victims were first raped before their 18th birthday."
    },
    {
        "question": "How much more likely were women raped as minors to report being raped as adults?",
        "source_passage": "18.3 percent of the women who said they were raped before age 18 also reported being raped since their 18th birthday, compared with 8.7 percent of the women who did not report being raped before age 18."
    },
    {
        "question": "What percentage of all female and male rape victims were raped by a current or former intimate partner?",
        "source_passage": "Overall, 43 percent of all female rape victims and 9 percent of all male victims were raped by some type of current or former intimate partner."
    },
    {
        "question": "Did most rapes committed against women by former spouses or cohabiting partners occur before or after the relationship ended?",
        "source_passage": "69.1 percent of the women who were raped by a former spouse or cohabiting partner said they were raped before the relationship ended"
    },
    {
        "question": "What percentage of female and male adult rape victims received mental health counseling as a result of their most recent rape?",
        "source_passage": "33 percent of the female rape victims and 24.2 percent of the male victims said they received counseling from a mental health professional as a direct result of their most recent rape"
    },
    {
        "question": "What percentage of female rape victims who obtained a restraining order said their rapist violated the order?",
        "source_passage": "Thirteen percent of the women who were raped since age 18 said they obtained a restraining order against their rapist (see \"Total\" column in exhibit 21). Of these women, 65.9 percent said their rapist violated the order."
    }
]

def evaluate_recall_at_5(eval_set, index, chunks, label=""):
    hits = 0
    failed_cases = []

    for i, item in enumerate(eval_set, 1):
        top_5 = retrieve_top_k(item["question"], index, chunks, k=5)
        target = clean_pdf_text(item["source_passage"]).lower()

        found = any(target in clean_pdf_text(c["text"]).lower() for c in top_5)
        if found:
            hits += 1
        else:
            failed_cases.append((i, item["question"], top_5[0]["text"][:200]))

    recall = (hits / len(eval_set)) * 100
    print(f"[{label}] Recall@5: {hits}/{len(eval_set)} ({recall:.2f}%)")
    return recall, failed_cases

small_recall, small_failures = evaluate_recall_at_5(eval_set, small_index, small_chunks, "Small Chunks (150w)")
large_recall, large_failures = evaluate_recall_at_5(eval_set, large_index, large_chunks, "Large Chunks (600w)")

print("\n--- Sample Failure Cases ---")
for q_num, q_text, top_retrieved in small_failures[:2]:
    print(f"Q{q_num}: {q_text}\nTop-1 Retrieved instead: {top_retrieved}...\n")

# --- Ek Question ke liye FAISS ke Top-6 Indices + Scores + Chunks Print Karna ---
def inspect_top_6_faiss(query, index, chunks, label=""):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, 6)  # k=6

    print(f"\n================ TOP-6 FAISS RESULTS ({label}) ================")
    print(f"Query: {query}\n")
    for rank, (faiss_idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        chunk = chunks[faiss_idx]
        print(f"Rank #{rank} | FAISS Index ID: {faiss_idx} | Page: {chunk['page']} | Cosine Score: {score:.4f}")
        print(f"Text Snippet: {chunk['text'][:220]}...\n")

# Example: Question 10 ke liye Top-6 FAISS indices print karo
inspect_top_6_faiss(eval_set[9]["question"], small_index, small_chunks, "Small Chunks")

In [ ]:
import torch
from transformers import pipeline

# 1. Local LLM pipeline load karo (float16 se fast aur kam memory mein chalega)
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)

def answer_question(query, index, chunks, k=3):
    # Top-k relevant chunks retrieve karo
    top_chunks = retrieve_top_k(query, index, chunks, k=k)
    context = "\n\n".join([f"[Page {c['page']}]: {c['text']}" for c in top_chunks])

    # Chat template format use karne se Qwen exact aur clean answer deta hai
    messages = [
        {
            "role": "system",
            "content": "You are a precise question-answering assistant. Answer the question using ONLY the provided context. If the answer is not in the context, say 'Not found'. Keep the answer concise and factual."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    # return_full_text=False se sirf model ka naya answer milega (prompt repeat nahi hoga)
    out = generator(
        messages,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )
    return out[0]["generated_text"].strip(), [c["page"] for c in top_chunks]

# 2. Saare 15 questions ke final answers generate aur print karo
print("================ FINAL RAG ANSWERS (ALL 15 QUESTIONS) ================\n")

rag_results = []

for i, item in enumerate(eval_set, 1):
    q = item["question"]
    ans, pages_used = answer_question(q, small_index, small_chunks, k=3)

    rag_results.append({
        "q_no": i,
        "question": q,
        "answer": ans,
        "pages": pages_used
    })

    print(f"Q{i}: {q}")
    print(f"Retrieved Pages: {pages_used}")
    print(f"Final Answer: {ans}")
    print("-" * 70)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

================ FINAL RAG ANSWERS (ALL 15 QUESTIONS) ================

Q1: How many women and men were interviewed in the National Violence Against Women Survey (NVAWS)?
Retrieved Pages: [5, 11, 42]
Final Answer: The National Violence Against Women Survey (NVAWS) interviewed 8,000 women and 8,005 men aged 18 or older.
----------------------------------------------------------------------
Q2: What proportion of women in the United States have been raped at some time in their lives according to NVAWS?
Retrieved Pages: [5, 13, 14]
Final Answer: According to the NVAWS, 1 out of every 6 women in the United States has been raped at some point in their lifetime.
----------------------------------------------------------------------
Q3: What were the primary reasons women chose not to report their adult rape victimization to the police?
Retrieved Pages: [8, 38, 36]
Final Answer: The primary reasons women chose not to report their adult rape victimization to the police included fear of their r

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q10: What percentage of female and male rape victims were first raped before their 18th birthday?
Retrieved Pages: [23, 23, 25]
Final Answer: According to the information provided, 54% of female victims and 71% of male victims were first raped before their 18th birthday.
----------------------------------------------------------------------
Q11: How much more likely were women raped as minors to report being raped as adults?
Retrieved Pages: [23, 24, 24]
Final Answer: Women were nearly equally likely to be raped as minors and adults. Specifically, 9.1% of all women surveyed said they were raped before their 18th birthday, which is about the same likelihood as those who were raped after turning 18.
----------------------------------------------------------------------
Q12: What percentage of all female and male rape victims were raped by a current or former intimate partner?
Retrieved Pages: [26, 26, 18]
Final Answer: 43% of all female rape victims and 9% of all male victims were raped 